# Approach 1: Template-Based Modeling (TBM) for Stanford RNA 3D Folding Part 2\n\n## What this notebook does\n1. Uses MMseqs2 to search test RNA sequences against the **entire PDB RNA database**\n2. For each hit: copies C1' coordinates from the template structure\n3. Generates 5 diverse predictions per target (top 5 template hits)\n4. Writes `submission.csv`\n\n## CRITICAL: This must run on Kaggle (not Colab)\nThe competition provides ~15,000 CIF files in `/kaggle/input/stanford-rna-3d-folding-2/PDB_RNA/`.\nThis is the **entire PDB RNA database**. You cannot replicate this on Colab.\n\n## Based on official sources\n- DasLab create_templates: https://github.com/DasLab/create_templates\n- Official baseline: https://www.kaggle.com/code/rhijudas/mmseqs2-3d-rna-template-identification\n- 1st place TBM: https://www.kaggle.com/code/jaejohn/rna-3d-folds-tbm-only-approach\n\n## RECOMMENDATION\nInstead of running THIS notebook, fork one of the official notebooks above.\nThey are battle-tested and optimized. This notebook is for UNDERSTANDING the pipeline.

In [ ]:
# ============================================================\n# CELL 1: Verify competition data is available\n# ============================================================\n# On Kaggle, competition data lives at:\n#   /kaggle/input/stanford-rna-3d-folding-2/\n#\n# Key files:\n#   PDB_RNA/          — ~15,000+ CIF files (the FULL PDB RNA database)\n#   PDB_RNA/pdb_seqres_NA.fasta — sequences of ALL RNA chains in PDB\n#   PDB_RNA/pdb_release_dates_NA.csv — release dates for temporal filtering\n#   test_sequences.csv — the test RNA sequences we must predict\n#   MSA/              — multiple sequence alignments (optional for TBM)\n# ============================================================\nimport os\nimport glob\n\n# Competition data root\nCOMP_DIR = '/kaggle/input/stanford-rna-3d-folding-2'\nPDB_RNA_DIR = os.path.join(COMP_DIR, 'PDB_RNA')\nTEST_CSV = os.path.join(COMP_DIR, 'test_sequences.csv')\nFASTA_DB = os.path.join(PDB_RNA_DIR, 'pdb_seqres_NA.fasta')\nDATES_CSV = os.path.join(PDB_RNA_DIR, 'pdb_release_dates_NA.csv')\n\n# Work directory (writable on Kaggle)\nWORK_DIR = '/kaggle/working/tbm_work'\nos.makedirs(WORK_DIR, exist_ok=True)\n\n# Verify\nprint('=== Competition data check ===')\nprint(f'Competition dir exists: {os.path.isdir(COMP_DIR)}')\nprint(f'PDB_RNA dir exists:     {os.path.isdir(PDB_RNA_DIR)}')\nprint(f'test_sequences.csv:     {os.path.isfile(TEST_CSV)}')\nprint(f'pdb_seqres_NA.fasta:    {os.path.isfile(FASTA_DB)}')\nprint(f'release_dates CSV:      {os.path.isfile(DATES_CSV)}')\n\n# Count CIF files\ncif_files = glob.glob(os.path.join(PDB_RNA_DIR, '*.cif')) + \\\n            glob.glob(os.path.join(PDB_RNA_DIR, '*.cif.gz'))\nprint(f'\\nCIF files in PDB_RNA:  {len(cif_files):,}')\nif cif_files:\n    print(f'First 3: {[os.path.basename(f) for f in sorted(cif_files)[:3]]}')\n    print(f'Last 3:  {[os.path.basename(f) for f in sorted(cif_files)[-3:]]}')

In [ ]:
# ============================================================\n# CELL 2: Load test sequences\n# ============================================================\nimport pandas as pd\n\ntest_df = pd.read_csv(TEST_CSV)\nprint(f'Test sequences: {len(test_df)}')\nprint(f'Columns: {list(test_df.columns)}')\nprint(test_df[['target_id', 'sequence']].head())

In [ ]:
# ============================================================\n# CELL 3: Install MMseqs2\n# ============================================================\n# MMseqs2 is a fast sequence search tool (like BLAST but much faster).\n# It finds RNA sequences in the PDB that are similar to our test sequences.\n#\n# For Kaggle SUBMISSION (no internet), you need to pre-install MMseqs2\n# as a Kaggle Dataset. For development runs with internet ON, this works:\n# ============================================================\n!apt-get update -qq && apt-get install -y -qq mmseqs2 2>/dev/null\n!mmseqs version\nprint('MMseqs2 installed successfully.')

In [ ]:
# ============================================================\n# CELL 4: Install BioPython\n# ============================================================\n!pip install biopython --quiet\nprint('BioPython installed.')

In [ ]:
# ============================================================\n# CELL 5: Prepare query FASTA from test sequences\n# ============================================================\n# MMseqs2 needs both query and target as FASTA files.\n# Query = our test sequences\n# Target = all known PDB RNA sequences (pdb_seqres_NA.fasta)\n# ============================================================\nQUERY_FASTA = os.path.join(WORK_DIR, 'test_query.fasta')\n\nwith open(QUERY_FASTA, 'w') as f:\n    for _, row in test_df.iterrows():\n        target_id = row['target_id']\n        sequence = row['sequence']\n        f.write(f'>{target_id}\\n{sequence}\\n')\n\nprint(f'Query FASTA written: {QUERY_FASTA}')\n!head -6 {QUERY_FASTA}

In [ ]:
# ============================================================\n# CELL 6: Build MMseqs2 databases and run search\n# ============================================================\n# This is the CORE step. MMseqs2 finds which PDB chains have\n# sequences similar to each test sequence.\n#\n# -s 7.5 = high sensitivity (finds more distant matches)\n# --search-type 3 = nucleotide search mode\n# ============================================================\nimport subprocess\nimport time\n\nQUERY_DB = os.path.join(WORK_DIR, 'queryDB')\nTARGET_DB = os.path.join(WORK_DIR, 'targetDB')\nRESULT_DB = os.path.join(WORK_DIR, 'resultDB')\nTMP_DIR = os.path.join(WORK_DIR, 'tmp')\nRESULT_FILE = os.path.join(WORK_DIR, 'Result.txt')\n\nos.makedirs(TMP_DIR, exist_ok=True)\n\nprint('Step 1/4: Building query database...')\nstart = time.time()\n!mmseqs createdb {QUERY_FASTA} {QUERY_DB} --dbtype 2\nprint(f'  Done in {time.time()-start:.1f}s')\n\nprint('\\nStep 2/4: Building target database from PDB sequences...')\nstart = time.time()\n!mmseqs createdb {FASTA_DB} {TARGET_DB} --dbtype 2\nprint(f'  Done in {time.time()-start:.1f}s')\n\nprint('\\nStep 3/4: Running MMseqs2 search (this takes a few minutes)...')\nstart = time.time()\n!mmseqs search {QUERY_DB} {TARGET_DB} {RESULT_DB} {TMP_DIR} \\\n    -s 7.5 --search-type 3 -e 10\nprint(f'  Done in {time.time()-start:.1f}s')\n\nprint('\\nStep 4/4: Converting results to readable format...')\n!mmseqs convertalis {QUERY_DB} {TARGET_DB} {RESULT_DB} {RESULT_FILE} \\\n    --format-output query,target,evalue,qstart,qend,tstart,tend,qaln,taln\n\n# Check results\nif os.path.exists(RESULT_FILE):\n    with open(RESULT_FILE) as f:\n        lines = f.readlines()\n    print(f'\\nMMseqs2 hits: {len(lines)}')\n    if lines:\n        print('First 3 hits:')\n        for line in lines[:3]:\n            parts = line.strip().split('\\t')\n            print(f'  Query={parts[0]} Target={parts[1]} E-value={parts[2]}')\nelse:\n    print('ERROR: No results file generated')

In [ ]:
# ============================================================\n# CELL 7: Clone DasLab create_templates and generate submission\n# ============================================================\n# The official create_templates_csv.py script does the heavy lifting:\n# - Parses MMseqs2 results\n# - Opens matching CIF files\n# - Aligns sequences and copies C1' coordinates\n# - Generates submission.csv in competition format\n#\n# Source: https://github.com/DasLab/create_templates\n# ============================================================\n\n# Clone the official repo\n!git clone https://github.com/DasLab/create_templates.git /kaggle/working/create_templates 2>/dev/null || \\\n    echo 'Already cloned'\n\n# Run the template creation script\n# Key arguments:\n#   -s : path to test_sequences.csv\n#   --mmseqs_results_file : MMseqs2 output\n#   --cif_dir : where the CIF files live (competition PDB_RNA/)\n#   --outfile : output submission CSV\n#   --max_templates : how many templates per target (we want 5 for the 5 predictions)\n\nOUTPUT_CSV = '/kaggle/working/submission.csv'\n\n!cd /kaggle/working/create_templates && python3 create_templates_csv.py \\\n    -s {TEST_CSV} \\\n    --mmseqs_results_file {RESULT_FILE} \\\n    --cif_dir {PDB_RNA_DIR} \\\n    --outfile {OUTPUT_CSV} \\\n    --max_templates 5\n\n# Verify output\nif os.path.exists(OUTPUT_CSV):\n    sub_df = pd.read_csv(OUTPUT_CSV)\n    print(f'\\nSubmission generated: {sub_df.shape}')\n    print(f'Columns: {list(sub_df.columns)}')\n    print(sub_df.head())\nelse:\n    print('ERROR: submission.csv not generated')

In [ ]:
# ============================================================\n# CELL 8: Verify submission and check coverage\n# ============================================================\nimport numpy as np\n\nif os.path.exists(OUTPUT_CSV):\n    sub_df = pd.read_csv(OUTPUT_CSV)\n    \n    # Check which targets got real templates vs dummy coords\n    targets = sub_df['ID'].str.rsplit('_', n=1).str[0].unique()\n    print(f'Total targets in submission: {len(targets)}')\n    \n    # Check coordinate range (dummy coords would be along a straight line)\n    for col_prefix in ['x_1', 'y_1', 'z_1']:\n        vals = sub_df[col_prefix]\n        print(f'{col_prefix}: min={vals.min():.1f} max={vals.max():.1f} std={vals.std():.1f}')\n    \n    print(f'\\nSubmission saved to: {OUTPUT_CSV}')\n    print('Ready for Kaggle submission!')

## What to do next\n\n1. **If this notebook ran on Kaggle**: Download `submission.csv` from the Output tab\n2. **Submit** to the competition\n3. **Check your score** on the leaderboard\n4. **For better results**: Fork `jaejohn/rna-3d-folds-tbm-only-approach` instead — it's the 1st place TBM approach\n5. **For best results**: Combine this with Approach 2 (RibonanzaNet distance prediction) for a hybrid submission